# 01. Imports

In [1]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
import torchvision.models as models
import cv2
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 02. Config

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224

# 03. Model

In [4]:
class CBMModel(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.features = nn.Sequential(*list(backbone.children())[:-1])

        self.shared = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.concept_head = nn.Linear(128, 4)   # NO, NC, CO, PSC
        self.presence_head = nn.Linear(128, 1)  # Cataract or not

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        shared = self.shared(x)
        concepts = self.concept_head(shared)
        presence = torch.sigmoid(self.presence_head(shared))
        return concepts, presence

# 04. Transform

In [5]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# 05. Severity Logic

In [6]:
def get_severity_label(score):
    if score < 1.0:
        return "Normal"
    elif score < 2.0:
        return "Mild"
    elif score < 3.0:
        return "Moderate"
    elif score < 4.0:
        return "Marked"
    elif score <= 5.0:
        return "Severe"
    else:
        return "Not in range of LOCS III"

def get_confidence(score):
    boundaries = np.array([1, 2, 3, 4])
    return float(np.min(np.abs(score - boundaries)))

def analyze_concepts(preds):
    preds = preds * 5.0  # denormalize
    names = ["NO", "NC", "CO", "PSC"]

    results = {}
    for i, name in enumerate(names):
        score = float(preds[i])
        results[name] = {
            "score": round(score, 2),
            "severity": get_severity_label(score),
            "confidence": round(get_confidence(score), 3)
        }

    return results

def compute_overall_severity(preds):
    preds = preds * 5.0

    weights = np.array([0.35, 0.35, 0.15, 0.15])
    weighted_score = np.sum(preds * weights)
    max_score = np.max(preds)

    final_score = 0.6 * max_score + 0.4 * weighted_score

    return {
        "score": round(float(final_score), 2),
        "severity": get_severity_label(final_score)
    }

def generate_full_report(preds):
    return {
        "concepts": analyze_concepts(preds),
        "overall": compute_overall_severity(preds),
        "cataract_type": detect_cataract_type(preds),
        "explanation": generate_text_explanation(preds),
        "treatment": generate_treatment_suggestion(preds)
    }

# 06. Grad-Cam Class

In [7]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.gradients = None
        self.activations = None

        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_image, target_index):
        self.model.zero_grad()

        concepts, _ = self.model(input_image)
        target = concepts[:, target_index]

        target.backward()

        gradients = self.gradients
        activations = self.activations

        weights = torch.mean(gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * activations, dim=1)

        cam = torch.relu(cam)
        cam = cam.squeeze().cpu().detach().numpy()

        cam = (cam - cam.min()) / (cam.max() + 1e-8)
        return cam

# 07. Heat Map

In [8]:
def overlay_cam_on_image(img, cam):
    cam = np.array(cam, dtype=np.float32)  # add this
    cam = cv2.resize(cam, (img.shape[1], img.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)

    overlay = heatmap * 0.4 + img
    overlay = np.clip(overlay / 255.0, 0, 1)

    return overlay

# 08. Explanations Generator

In [9]:
def explain_image(image_path, model_path):
    model = CBMModel().to(DEVICE)
    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    image = Image.open(image_path).convert("RGB")
    original = np.array(image)

    input_tensor = transform(image).unsqueeze(0).to(DEVICE)

    # ===== Prediction =====
    with torch.no_grad():
        concepts, presence = model(input_tensor)
        presence_score = presence.squeeze().item()

        if presence_score < 0.5:
            print("No cataract detected.")
            return

        preds = concepts.squeeze().cpu().numpy()
        preds = np.clip(preds, 0, 1)

    # ===== Get dominant concept =====
    target_index = get_dominant_concept_index(preds)

    # ===== Grad-CAM =====
    target_layer = model.features[-1]
    gradcam = GradCAM(model, target_layer)

    cam = gradcam.generate(input_tensor, target_index)
    overlay = overlay_cam_on_image(original, cam)

    concept_names = ["NO", "NC", "CO", "PSC"]

    plt.figure(figsize=(5, 5))
    plt.imshow(overlay)
    plt.title(f"Dominant Concept: {concept_names[target_index]}")
    plt.axis("off")
    plt.show()

#





In [10]:
def detect_cataract_type(preds):
    preds = preds * 5.0  # denormalize

    NO, NC, CO, PSC = preds

    # Nuclear score (combined)
    nuclear_score = (NO + NC) / 2.0

    # Individual type strengths
    type_scores = {
        "Nuclear": nuclear_score,
        "Cortical": CO,
        "PSC": PSC
    }

    # Find dominant type
    detected_type = max(type_scores, key=type_scores.get)
    confidence = type_scores[detected_type] / 5.0  # normalize 0–1

    return {
        "type": detected_type,
        "confidence": round(float(confidence), 3),
        "all_scores": {k: round(float(v), 2) for k, v in type_scores.items()}
    }

def generate_text_explanation(preds):
    preds = preds * 5.0
    NO, NC, CO, PSC = preds

    explanation = []

    # Nuclear reasoning
    if NO > 2.5 or NC > 2.5:
        explanation.append(
            f"High Nuclear Opalescence (NO={NO:.2f}) and/or Nuclear Color (NC={NC:.2f}) "
            f"indicate increased opacity in the central lens region, consistent with Nuclear Cataract."
        )

    # Cortical reasoning
    if CO > 2.5:
        explanation.append(
            f"Elevated Cortical Opacity (CO={CO:.2f}) suggests peripheral spoke-like opacities, "
            f"which are characteristic of Cortical Cataract."
        )

    # PSC reasoning
    if PSC > 2.5:
        explanation.append(
            f"High Posterior Subcapsular score (PSC={PSC:.2f}) indicates opacity near the back of the lens, "
            f"which strongly affects central vision and is typical of PSC cataracts."
        )

    # Mild / no strong signal
    if len(explanation) == 0:
        explanation.append(
            "All cataract-related features are within low ranges, suggesting no significant cataract formation."
        )

    return explanation

def generate_treatment_suggestion(preds):
    preds = preds * 5.0
    NO, NC, CO, PSC = preds

    max_score = max(NO, NC, CO, PSC)

    # Basic clinical decision rules
    if max_score < 2.0:
        return {
            "action": "No immediate treatment required",
            "recommendation": "Regular monitoring is advised. Maintain eye health and routine check-ups."
        }

    elif max_score < 3.0:
        return {
            "action": "Non-surgical management",
            "recommendation": "Consider updating eyeglass prescription, improving lighting conditions, and periodic monitoring."
        }

    elif max_score < 4.0:
        return {
            "action": "Clinical evaluation recommended",
            "recommendation": "Consult an ophthalmologist. Cataract progression may begin affecting daily activities."
        }

    else:
        return {
            "action": "Surgical intervention likely required",
            "recommendation": "Cataract surgery should be considered, especially if vision impairment affects quality of life."
        }


def get_dominant_concept_index(preds):
    preds = preds * 5.0
    NO, NC, CO, PSC = preds

    nuclear_score = max(NO, NC)

    scores = [nuclear_score, CO, PSC]
    idx = int(np.argmax(scores))

    if idx == 0:
        return 0  # NO (nuclear)
    elif idx == 1:
        return 2  # CO
    else:
        return 3  # PSC

def predict_image(image_path, model_path):
    # Load model
    model = CBMModel().to(DEVICE)
    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()

    # Load image
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(DEVICE)

    # Predict
    with torch.no_grad():
        concepts, presence = model(image)
        preds = concepts.squeeze().cpu().numpy()
        preds = np.clip(preds, 0, 1)

    # Generate report
    report = generate_full_report(preds)

    return report

# 10. Run Method

In [11]:
if __name__ == "__main__":
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import tempfile, os

    uploader = widgets.FileUpload(
        accept='.jpg,.jpeg,.png,.bmp,.tiff,.tif',
        multiple=False,
        description='Upload Image'
    )

    run_button = widgets.Button(
        description='Analyze Image',
        button_style='primary',
        disabled=True
    )

    output = widgets.Output()

    def on_upload_change(change):
        if uploader.value:
            run_button.disabled = False
            with output:
                clear_output()
                file_info = list(uploader.value.values())[0]
                filename = file_info['metadata']['name']
                print(f"File ready: {filename}")

    def on_run_clicked(b):
        with output:
            clear_output()
            file_info = list(uploader.value.values())[0]
            filename = file_info['metadata']['name']
            file_bytes = file_info['content']

            ext = os.path.splitext(filename)[1]
            with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
                tmp.write(bytes(file_bytes))
                image_path = tmp.name

            model_path = "/content/drive/MyDrive/IIT/Academic/4th Year/FYP/IPD/Code Base/best_model.pth"

            print(f"Analyzing: {filename}\n")
            result = predict_image(image_path, model_path)
            explain_image(image_path, model_path)

            print("===== Cataract Analysis Report =====")
            for k, v in result["concepts"].items():
                print(f"{k}: {v}")
            print("\nOverall:", result["overall"])
            print("\nCataract Type:", result["cataract_type"])
            print("\nExplanation:")
            for line in result["explanation"]:
              print("-", line)
            print("\nTreatment Suggestion:")
            print("Action:", result["treatment"]["action"])
            print("Recommendation:", result["treatment"]["recommendation"])

            os.unlink(image_path)

    uploader.observe(on_upload_change, names='value')
    run_button.on_click(on_run_clicked)

    display(widgets.VBox([uploader, run_button, output]))